## Automated Frame Extraction & Image Enhancement

#### This notebook focuses on the pre-processing stage of the road defect detection pipeline. It automates the process of extracting key frames from raw video data to reduce computational overhead while enhancing critical features for analysis.

In [4]:
import cv2
import numpy as np

# Load Video
cap = cv2.VideoCapture("vid.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0
saved_count = 0

print(f"Enhancement started... FPS: {fps}")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # using this logic, we can process one image every second
    if int(frame_count % int(fps)) == 0:
        
        # convert image to gray scale to reduce the computation power and time
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Power-Law (Gamma) Transformation
        # Used to increase contrast by darkening pixels.
        # This makes road cracks appear much darker and easier to detect.
        gamma = 2.0
        gamma_corrected = np.array(255 * (gray / 255) ** gamma, dtype='uint8')
        
        # Median Filtering (Noise Reduction)
        # Used to remove small dots, sand, and salt-and-pepper noise.
        # This smooths the road surface while preserving the sharp edges of the cracks.
        median_filtered = cv2.medianBlur(gamma_corrected, 5)
    
        # Laplacian Sharpening
        # Used to detect intensity changes and highlight the edges of the cracks.
        # It makes the crack boundaries appear as bright white lines against the background.
        laplacian = cv2.Laplacian(median_filtered, cv2.CV_64F)
        sharpened_edges = np.uint8(np.absolute(laplacian))

        # Min-Max Normalization (Contrast Stretching)
        # Used to expand the pixel range to the full 0-255 scale.
        # This makes the faint crack edges much brighter and more distinct.
        stretched = cv2.normalize(sharpened_edges, None, 0, 255, cv2.NORM_MINMAX)
        
        # Image Subtraction
        # Used to darken the crack areas by subtracting the bright edges from the filtered image.
        # Since the cracks are now represented by white pixels (high values) in the 'stretched' image,
        # subtracting them from the original makes those specific areas turn black.
        final_enhanced = cv2.subtract(median_filtered, stretched)
        
        # Final Normalization (Optional)
        # Used to stretch the pixel intensity to the full 0-255 range.
        # This maximizes the contrast, making the background brighter and the cracks darker.
        final_enhanced = cv2.normalize(final_enhanced, None, 0, 255, cv2.NORM_MINMAX)

        # Save to folder
        cv2.imwrite(f"saved_enhanced/frame_{saved_count}.jpg", final_enhanced)
        saved_count += 1
        
    frame_count += 1

cap.release()
print(f"Step 1 Complete! {saved_count} frames saved in 'saved_enhanced' folder.")

Enhancement started... FPS: 29.964432218956098
Step 1 Complete! 17 frames saved in 'saved_enhanced' folder.
